# 02 · What a session looks like

Prompts per session, how long a session lasts, whether students come back, and when in the day they
ask.

Medians lead, means follow. With a few hundred sessions a single long one moves a mean visibly and a
median not at all, and the median is the number that describes a typical student.

In [1]:
# Put the analysis package on the path no matter where Jupyter was started.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "analysis" / "bloombot_analysis").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "analysis"))

import pandas as pd

from bloombot_analysis import charts, load, metrics, privacy, report, sessions, topics
from bloombot_analysis.config import CONFIG, SURFACE_LABELS, TOPICS

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("as of:", CONFIG.as_of)
print("legacy db :", CONFIG.legacy_db, "(exists)" if CONFIG.legacy_db.exists() else "(missing)")
print("current db:", CONFIG.current_db, "(exists)" if CONFIG.current_db.exists() else "(missing)")
print("output    :", CONFIG.out_dir)

as of: 2026-09-25
legacy db : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/legacy.db (exists)
current db: /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/current.db (exists)
output    : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/out


In [2]:
sessions_df = pd.read_csv(
    CONFIG.data_path("sessions.csv"), parse_dates=["started_at", "ended_at", "week"]
)
sessions_df["date"] = pd.to_datetime(sessions_df["date"]).dt.date
messages_df = pd.read_csv(CONFIG.data_path("messages.csv"), parse_dates=["ts", "week"])
print(len(sessions_df), "sessions,", len(messages_df), "messages")

259 sessions, 1584 messages


## Prompts per session

Only student messages are counted. Counting the bot's replies too would double every length and make
sessions look twice as deep as they are.

In [3]:
summary = sessions.usage_summary(sessions_df)
prompts_fig = charts.histogram(
    sessions_df["prompts"],
    "prompts_per_session",
    "Prompts per session",
    "Student messages in one session",
)
print(
    f"median {summary['median_prompts']:.0f} · mean {summary['mean_prompts']:.1f} · "
    f"IQR {summary['iqr_prompts'][0]:.0f}–{summary['iqr_prompts'][1]:.0f} · n = {summary['sessions']}"
)
sessions_df["prompts"].describe()

median 3 · mean 3.1 · IQR 2–4 · n = 259


count    259.000000
mean       3.057915
std        1.644973
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max       12.000000
Name: prompts, dtype: float64

## Session length in minutes

A one-prompt session has a duration near zero by construction (it is a single message), so this
distribution is strongly floor-heavy. That is a fact about how students use the bot — most visits are
one question — not an artefact to be smoothed away.

In [4]:
duration_fig = charts.histogram(
    sessions_df["duration_minutes"],
    "session_duration",
    "Session length (minutes)",
    "Minutes from first to last message",
    bins=20,
)
print(
    f"median {summary['median_duration']:.1f} min · "
    f"IQR {summary['iqr_duration'][0]:.1f}–{summary['iqr_duration'][1]:.1f} min"
)
sessions_df["duration_minutes"].describe()

median 13.8 min · IQR 5.3–22.1 min


count    259.000000
mean      15.857722
std       13.764430
min        0.066667
25%        5.291667
50%       13.750000
75%       22.133333
max       79.116667
Name: duration_minutes, dtype: float64

## Coming back

Reported as counts rather than a retention curve: at this sample size a curve implies a precision the
data does not have.

In [5]:
returns = sessions.return_rate(sessions_df)
returns

{'students': 54,
 'returned': 38,
 'share': 0.7037037037037037,
 'median_active_days': 3.0}

## Hour of day

The question every audience asks about an always-on assistant: are students using it outside the
hours a human would answer?

In [6]:
hours = sessions.hour_of_day(sessions_df)
hour_fig = charts.column_bar(
    [f"{h:02d}" for h in hours["hour"]],
    hours["sessions"].tolist(),
    "sessions_by_hour",
    "Sessions by hour of day started (local time)",
    "Sessions",
)
after_hours = int(hours[(hours["hour"] >= 18) | (hours["hour"] < 8)]["sessions"].sum())
total_sessions = int(hours["sessions"].sum())
print(f"{after_hours} of {total_sessions} sessions start between 18:00 and 08:00")

160 of 259 sessions start between 18:00 and 08:00


## Sensitivity to the session gap

The 30-minute cut is a convention. Re-deriving the headline numbers at 15 and 60 minutes is what
turns it from an arbitrary choice into a stated, checkable one. If these barely move, the report says
so in one line.

In [7]:
sensitivity = sessions.gap_sensitivity(messages_df)
sensitivity_path = CONFIG.data_path("gap_sensitivity.csv")
sensitivity.to_csv(sensitivity_path, index=False)
sensitivity

,gap_minutes,sessions,median_prompts,median_duration_minutes
0,15,260,3.0,13.625
1,30,259,3.0,13.750
2,60,259,3.0,13.750


In [8]:
metrics.update("shape", {
    **summary,
    "returns": returns,
    "after_hours_sessions": after_hours,
    "total_sessions": total_sessions,
    "sensitivity": sensitivity.to_dict(orient="records"),
    "single_prompt_sessions": int((sessions_df["prompts"] == 1).sum()),
    "long_sessions": int((sessions_df["prompts"] >= 5).sum()),
    "figures": {
        "prompts": prompts_fig.name,
        "duration": duration_fig.name,
        "hours": hour_fig.name,
    },
})
print("ok")

ok
